In [23]:
# Install PySCF and GPU4PySCF (CUDA 12.x version for Colab's T4 GPU)
# cuTENSOR and CuPy are also installed for maximum tensor contraction acceleration.
!pip install pyscf gpu4pyscf-cuda12x cupy-cuda12x pyscf-dispersion geometric

In [24]:
# Must be set BEFORE any gpu4pyscf imports
import gpu4pyscf.__config__ as gpu4pyscf_config
gpu4pyscf_config.num_devices = 2

# Verify both GPUs
import cupy as cp
for i in range(2):
    with cp.cuda.Device(i):
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU {i}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

GPU 0: 15.1 GB free / 15.6 GB total
GPU 1: 14.7 GB free / 15.6 GB total


In [25]:
import os

xyz_filename = "/kaggle/input/datasets/pcsciprav/xtbopt2/xtbopt.xyz"

print(f"Successfully loaded: {xyz_filename}")

Successfully loaded: /kaggle/input/datasets/pcsciprav/xtbopt2/xtbopt.xyz


In [26]:
import cupy as cp

# Verify both GPUs are visible
n_gpus = cp.cuda.runtime.getDeviceCount()
print(f"GPUs available: {n_gpus}")
for i in range(n_gpus):
    with cp.cuda.Device(i):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(f"  GPU {i}: {props['name'].decode()}, {props['totalGlobalMem'] / 1e9:.1f} GB")

GPUs available: 2
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB


In [27]:
import pyscf
from pyscf import gto
from gpu4pyscf.dft import rks
from pyscf.geomopt.geometric_solver import optimize

# 2. Initialize the Molecule from the uploaded xyz file
with open(xyz_filename, 'r') as f:
    lines = f.readlines()

# XYZ format: line 0 = atom count, line 1 = comment, line 2+ = atoms
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]  # skip the xtb comment line
xyz_contents = ''.join(atom_lines)

mol = gto.M(
    atom = xyz_contents,
    basis = 'def2-QZVPPD',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol.max_memory = 9000
print(f"\nMolecule initialized: {mol.natm} atoms, {mol.nao} atomic orbitals.")

# 3. Configure the GPU-Accelerated DFT Calculation
mf = rks.RKS(mol)
mf.xc = 'b3lyp-d3bj'
mf = mf.density_fit()

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # Enable both T4s

# Free all VRAM before optimization
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

# Lower memory usage per SCF cycle
mf.max_memory = 9000
mf.conv_tol = 1e-8
mf.conv_tol_grad = 1e-5  # slightly relaxed for optimization steps

# 4. RUN GEOMETRY OPTIMIZATION
cp.get_default_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated Geometry Optimization...")
def save_callback(envs):
    mol_envs = envs['mol']
    mol_envs.tofile('/kaggle/working/dft_optimized_checkpoint.xyz')

mol_eq = optimize(mf, maxsteps=50, trust=0.1, tmax=0.1, callback=save_callback)
# 5. Save the results
output_file = 'dft_optimized.xyz'
mol_eq.tofile(output_file)

print("\n" + "="*50)
print(f"Optimization complete! New coordinates saved to: {output_file}")
print("="*50)

System: uname_result(system='Linux', node='a1dce743e31e', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Wed Mar 25 16:02:10 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 10
[INPUT] num. electrons = 74
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole

geometric-optimize called with the following command line:
/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-2df31b19-bfb0-4ce2-b65a-50ec299ebb58.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **              ********    


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.795395  -0.335744  -0.126194    0.000000  0.000000  0.000000
   O   0.708814  -1.458521  -0.486757    0.000000  0.000000  0.000000
   O  -0.024678   0.476678   0.134659    0.000000  0.000000  0.000000
   N   2.145465   0.100443   0.013870    0.000000  0.000000  0.000000
   N   2.847140   1.256983   0.385239    0.000000  0.000000 -0.000000
   N   2.568608   2.588153   0.812708    0.000000  0.000000  0.000000
   O   3.546348   3.221322   1.015884   -0.000000  0.000000  0.000000
   O   1.419505   2.857824   0.899210    0.000000  0.000000  0.000000
   N   3.995004   0.554528   0.159651    0.000000  0.000000  0.000000
   N   3.358079  -0.495295  -0.177438   -0.000000  0.000000  0.000000

WARN: Mole.unit (angstrom) is changed to Bohr

New geometry
   1 N      0.795395379671  -0.335743668896  -0.126194144063 AA    1.503079428323  -0.634463582271  -0.23847237

Step    0 : Gradient = 2.042e-02/2.733e-02 (rms/max) Energy = -629.2733451925
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.30000e-02 ... 1.13590e+00 1.14167e+00 1.14174e+00



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.786791  -0.351734  -0.131786   -0.008605 -0.015990 -0.005592
   O   0.706675  -1.487219  -0.495795   -0.002139 -0.028697 -0.009038
   O  -0.059970   0.451209   0.125253   -0.035292 -0.025468 -0.009406
   N   2.141904   0.102175   0.014066   -0.003562  0.001732  0.000196
   N   2.844733   1.261456   0.386450   -0.002407  0.004473  0.001210
   N   2.578968   2.605478   0.818152    0.010359  0.017325  0.005444
   O   3.574670   3.234164   1.022111    0.028323  0.012842  0.006227
   O   1.423586   2.895879   0.910794    0.004081  0.038056  0.011585
   N   4.002523   0.559515   0.161217    0.007519  0.004988  0.001566
   N   3.359344  -0.501281  -0.179468    0.001265 -0.005986 -0.002030
New geometry
   1 N      0.786790768817  -0.351734122519  -0.131786043385 AA    1.486819070401  -0.664681160226  -0.249039529037 Bohr
   2 O      0.706675375664  -1.4872185

...


Geometry optimization cycle 9
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.790011  -0.340046  -0.131034   -0.000012 -0.000306 -0.000099
   O   0.709639  -1.478844  -0.496505    0.001150  0.000015 -0.001296
   O  -0.049145   0.474153   0.128276   -0.000653 -0.000995 -0.000123
   N   2.146570   0.105034   0.012986   -0.000488  0.000776 -0.000128
   N   2.845488   1.257885   0.385370   -0.000507  0.000712 -0.000047
   N   2.570525   2.598551   0.816684    0.000197  0.000906 -0.000618
   O   3.564782   3.233612   1.028830    0.000488 -0.000418  0.001896
   O   1.410468   2.883567   0.902782    0.000429  0.001830 -0.000916
   N   4.007701   0.561061   0.165373   -0.000480  0.000125  0.002535
   N   3.360902  -0.504802  -0.181733   -0.000516  0.001311 -0.001306
New geometry
   1 N      0.790011263716  -0.340045539861  -0.131034369747 AA    1.492904923745  -0.642592940216  -0.247619071727 Bohr
   2 O      0.709639228903  -1.4788443

Step    8 : Displace = 1.445e-03/1.972e-03 (rms/max) Trust = 2.479e-03 (+) Grad = 3.125e-04/5.074e-04 (rms/max) E (change) = -629.2755118500 (+1.307e-06) Quality = -1.125
Rejecting step - quality is lower than -1.0
Hessian Eigenvalues: 2.29158e-02 2.30001e-02 2.41599e-02 ... 1.14033e+00 1.15017e+00 1.19960e+00



Geometry optimization cycle 10
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.789978  -0.340020  -0.131110   -0.000033  0.000026 -0.000076
   O   0.709284  -1.479041  -0.495831   -0.000355 -0.000197  0.000674
   O  -0.049019   0.474405   0.128043    0.000126  0.000253 -0.000232
   N   2.146671   0.104709   0.013124    0.000101 -0.000325  0.000138
   N   2.845650   1.257570   0.385459    0.000162 -0.000315  0.000089
   N   2.570498   2.598114   0.817151   -0.000027 -0.000437  0.000467
   O   3.564825   3.233647   1.027573    0.000043  0.000035 -0.001257
   O   1.410407   2.882919   0.903603   -0.000061 -0.000648  0.000821
   N   4.007849   0.561194   0.163811    0.000148  0.000133 -0.001563
   N   3.361010  -0.505447  -0.180740    0.000108 -0.000645  0.000993
New geometry
   1 N      0.789978085726  -0.340019773929  -0.131110030142 AA    1.492842226431  -0.642544249663  -0.247762049152 Bohr
   2 O      0.709284282640  -1.479041

Step    9 : Displace = 7.808e-04/1.142e-03 (rms/max) Trust = 7.226e-04 (x) Grad = 5.762e-05/9.613e-05 (rms/max) E (change) = -629.2755135294 (-3.725e-07) Quality = 0.507
Hessian Eigenvalues: 2.29158e-02 2.30001e-02 2.41599e-02 ... 1.14033e+00 1.15017e+00 1.19960e+00
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 187.067 seconds



Optimization complete! New coordinates saved to: dft_optimized.xyz


In [28]:
import os
for f in os.listdir('/kaggle/working'):
    print(f)
import glob
print(glob.glob('/kaggle/working/*.xyz'))
print(glob.glob('/kaggle/working/**/*.xyz', recursive=True))

dft_optimized.xyz
dft_optimized_final.xyz
.virtual_documents
hessian_matrix.npy
dft_optimized_checkpoint.xyz
['/kaggle/working/dft_optimized.xyz', '/kaggle/working/dft_optimized_final.xyz', '/kaggle/working/dft_optimized_checkpoint.xyz']
['/kaggle/working/dft_optimized.xyz', '/kaggle/working/dft_optimized_final.xyz', '/kaggle/working/dft_optimized_checkpoint.xyz']


In [29]:
# Cell 4: Vibrational Frequencies and Thermochemistry
from gpu4pyscf.dft import rks
from gpu4pyscf.hessian import rks as gpu_hessian
from pyscf.hessian import thermo
import cupy as cp
import numpy as np
from pyscf import gto
import numpy as np

with open('/kaggle/working/dft_optimized.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = 'def2-QZVPPD',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 10000

print("1. Setting up new SCF for the optimized geometry...")
mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'b3lyp-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
e_opt = mf_eq.kernel()

print("\n2. Calculating analytical Hessian (Frequencies) on the GPU...")
hess = gpu_hessian.Hessian(mf_eq)
hess_matrix = hess.kernel()

print("\n3. Calculating Thermochemistry (298.15 K, 1 atm)...")
freq_info = thermo.harmonic_analysis(mol_eq, hess_matrix)
thermo_data = thermo.thermo(mf_eq, freq_info['freq_au'], 298.15, 101325.0)

print("\n--- Thermochemistry Summary ---")
thermo.dump_thermo(mol_eq, thermo_data)

# ==========================================
# 4. SMART MEMORY MANAGEMENT & BACKUP
# ==========================================
print("\n4. Saving data to disk and purging VRAM...")

# Step A: Save the massive Hessian matrix to a file so we don't lose it
np.save('hessian_matrix.npy', np.asarray(hess_matrix))
print("   -> Hessian securely saved to 'hessian_matrix.npy'")

# Step B: Explicitly delete the giant objects from Python's memory
del hess
#del hess_matrix
del freq_info
del thermo_data

# Step C: Force the GPU to empty the trash now that Python let go of the variables
cp.get_default_memory_pool().free_all_blocks()
print("   -> GPU VRAM successfully flushed!")

print("\n" + "="*50)
print("FREQUENCY & THERMO CALCULATION COMPLETE")
print("="*50)

System: uname_result(system='Linux', node='a1dce743e31e', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Wed Mar 25 16:05:17 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 10
[INPUT] num. electrons = 74
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole

In [30]:
import numpy as np
import cupy as cp

print("="*50)
print("HESSIAN / EIGENVALUE SUMMARY")
print("="*50)

H = hess_matrix
if isinstance(H, cp.ndarray):
    H = cp.asnumpy(H)

# block Hessian (natm,natm,3,3) -> full matrix
natm = H.shape[0]
Hfull = H.transpose(0,2,1,3).reshape(3*natm, 3*natm)

w = np.linalg.eigvalsh(Hfull)

print("Shape of full Hessian:", Hfull.shape)
print("Smallest 20 eigenvalues:")
print(w[:20])
print("Largest 10 eigenvalues:")
print(w[-10:])

tol = 1e-8
nneg = np.sum(w < -tol)
nzero = np.sum(np.abs(w) <= tol)
npos = np.sum(w > tol)

print("Negative eigenvalues:", nneg)
print("Near-zero eigenvalues:", nzero)
print("Positive eigenvalues:", npos)
print("="*50)
print("SOFTEST MODE ATOMIC PARTICIPATION")
print("="*50)

eigvals, eigvecs = np.linalg.eigh(Hfull)

for mode in range(min(5, len(eigvals))):
    vec = eigvecs[:, mode].reshape(natm, 3)
    atom_amp = np.linalg.norm(vec, axis=1)
    top = np.argsort(atom_amp)[::-1][:10]
    print(f"\nMode {mode}  eigenvalue = {eigvals[mode]: .8f}")
    for i in top:
        print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  amplitude = {atom_amp[i]:.6f}")

HESSIAN / EIGENVALUE SUMMARY
Shape of full Hessian: (30, 30)
Smallest 20 eigenvalues:
[-7.30747848e-02 -4.48122205e-02 -3.04170961e-02 -2.97323393e-02
 -2.80453964e-02 -1.11756774e-02 -1.03202508e-02 -8.27209723e-03
 -4.55413288e-03 -3.89868008e-04  5.28753530e-05  3.66165627e-04
  6.25241805e-02  6.45691515e-02  1.03485219e-01  1.46261410e-01
  1.71413889e-01  1.93718005e-01  2.07403086e-01  3.20596310e-01]
Largest 10 eigenvalues:
[0.36173303 0.59396681 0.66089674 0.88395041 0.90092912 0.95595879
 0.9968056  1.16279939 1.67720566 1.71467234]
Negative eigenvalues: 10
Near-zero eigenvalues: 0
Positive eigenvalues: 20
SOFTEST MODE ATOMIC PARTICIPATION

Mode 0  eigenvalue = -0.07307478
Atom  4 N   amplitude = 0.644991
Atom  3 N   amplitude = 0.635741
Atom  9 N   amplitude = 0.252653
Atom  8 N   amplitude = 0.248934
Atom  2 O   amplitude = 0.155645
Atom  7 O   amplitude = 0.154198
Atom  0 N   amplitude = 0.050824
Atom  5 N   amplitude = 0.049446
Atom  6 O   amplitude = 0.023289
Atom  1 O  

In [31]:
# Cell 5: Excited States (TD-DFT) for UV-Vis
from gpu4pyscf.tdscf import rks as gpu_tdscf
import cupy as cp
from pyscf import gto
from gpu4pyscf.dft import rks

with open('dft_optimized_checkpoint.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = '3-21g',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 10000

mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'b3lyp-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
mf_eq.kernel()
print("Cleaning up GPU VRAM from previous calculations...")
# ==========================================
# CRITICAL VRAM CLEANUP
# Forces the GPU to release cached memory from the Hessian calc
# ==========================================
cp.get_default_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated TD-DFT...")
print("Calculating the first 5 singlet excited states...\n")

# Initialize Time-Dependent DFT using our optimized molecule's SCF object (mf_eq)
td = gpu_tdscf.TDDFT(mf_eq)
td.nstates = 5       # Number of excited states to calculate
td.singlet = True    # Look for singlet-singlet transitions

# ==========================================
# VRAM OPTIMIZATION FOR TD-DFT
# Restricts the Davidson solver's maximum memory footprint
# ==========================================
td.max_space = 15

# Run the calculation
td.kernel()

print("\n" + "="*50)
print("TD-DFT EXCITATION ENERGIES:")
print("="*50)
# This will print the transition energies (in eV), wavelengths (in nm), and oscillator strengths (intensity)
td.analyze()

System: uname_result(system='Linux', node='a1dce743e31e', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Wed Mar 25 16:05:30 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 10
[INPUT] num. electrons = 74
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole

In [32]:
print("="*50)
print("TDDFT SUMMARY")
print("="*50)

print("Excitation energies (eV):")
print(cp.asnumpy(td.e) * 27.211386245988 if hasattr(td.e, "get") or "cupy" in str(type(td.e)).lower() else td.e * 27.211386245988)

print("\nOscillator strengths:")
try:
    print(td.oscillator_strength())
except Exception as e:
    print("Could not get oscillator strengths directly:", e)

TDDFT SUMMARY
Excitation energies (eV):
[4.62016126 4.72685153 4.94880296 5.07635757 5.275351  ]

Oscillator strengths:
[7.31597426e-04 3.82784645e-07 3.70750520e-03 1.15952064e-07
 2.13437455e-02]


In [33]:
import cupy as cp
from pyscf.scf import hf

print("="*50)
print("1. MOLECULAR DIPOLE MOMENT")
print("="*50)
dipole = mf_eq.dip_moment()

print("\n" + "="*50)
print("2. MULLIKEN ATOMIC CHARGES")
print("="*50)

dm = mf_eq.make_rdm1()
s = mol_eq.intor('int1e_ovlp')

# Convert GPU array to CPU NumPy array
dm = cp.asnumpy(dm)

mulliken_pop, mulliken_chg = hf.mulliken_pop(mol_eq, dm, s=s)

print("\nAtom | Charge")
print("-" * 20)
for i, charge in enumerate(mulliken_chg):
    atom_symbol = mol_eq.atom_symbol(i)
    print(f"{i:2d} {atom_symbol:2s} | {charge: .4f}")

1. MOLECULAR DIPOLE MOMENT
Dipole moment(X, Y, Z, Debye): -0.52760,  0.28951,  0.09228

2. MULLIKEN ATOMIC CHARGES
 ** Mulliken pop  **
pop of  0 N 1s            1.98584
pop of  0 N 2s            0.44606
pop of  0 N 3s            0.89314
pop of  0 N 2px           0.59646
pop of  0 N 2py           0.68315
pop of  0 N 2pz           0.62013
pop of  0 N 3px           0.42720
pop of  0 N 3py           0.23881
pop of  0 N 3pz           0.61342
pop of  1 O 1s            1.98626
pop of  1 O 2s            0.44918
pop of  1 O 3s            1.57795
pop of  1 O 2px           0.85479
pop of  1 O 2py           0.58429
pop of  1 O 2pz           0.66310
pop of  1 O 3px           0.87891
pop of  1 O 3py           0.52075
pop of  1 O 3pz           0.73377
pop of  2 O 1s            1.98629
pop of  2 O 2s            0.44831
pop of  2 O 3s            1.58323
pop of  2 O 2px           0.71225
pop of  2 O 2py           0.71903
pop of  2 O 2pz           0.67304
pop of  2 O 3px           0.67423
pop of  2 O 3p

In [34]:
print("="*50)
print("ATOM INDEX MAP")
print("="*50)

for i in range(mol_eq.natm):
    x, y, z = mol_eq.atom_coord(i)
    print(f"{i:2d} {mol_eq.atom_symbol(i):2s}  {x: .6f} {y: .6f} {z: .6f}")

ATOM INDEX MAP
 0 N    1.492842 -0.642544 -0.247762
 1 O    1.340353 -2.794982 -0.936985
 2 O   -0.092633  0.896496  0.241967
 3 N    4.056620  0.197871  0.024801
 4 N    5.377500  2.376462  0.728411
 5 N    4.857537  4.909724  1.544192
 6 O    6.736544  6.110708  1.941832
 7 O    2.665283  5.447927  1.707562
 8 N    7.573737  1.060503  0.309558
 9 N    6.351388 -0.955157 -0.341549


In [35]:
import numpy as np

print("="*50)
print("CHARGE SUMMARY")
print("="*50)

charges = np.array(mulliken_chg)

print("Min charge:", charges.min())
print("Max charge:", charges.max())
print("Mean charge:", charges.mean())
print("Std dev:", charges.std())

print("\nTop 10 most positive atoms")
for i in np.argsort(charges)[::-1][:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

print("\nTop 10 most negative atoms")
for i in np.argsort(charges)[:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

CHARGE SUMMARY
Min charge: -0.2664318092382487
Max charge: 0.49579682394458935
Mean charge: -4.085620730620576e-15
Std dev: 0.27589244564925935

Top 10 most positive atoms
Atom  0 N   q =  0.495797
Atom  5 N   q =  0.495755
Atom  8 N   q =  0.038097
Atom  9 N   q =  0.038007
Atom  3 N   q = -0.018346
Atom  4 N   q = -0.018375
Atom  1 O   q = -0.248999
Atom  6 O   q = -0.249087
Atom  2 O   q = -0.266416
Atom  7 O   q = -0.266432

Top 10 most negative atoms
Atom  7 O   q = -0.266432
Atom  2 O   q = -0.266416
Atom  6 O   q = -0.249087
Atom  1 O   q = -0.248999
Atom  4 N   q = -0.018375
Atom  3 N   q = -0.018346
Atom  9 N   q =  0.038007
Atom  8 N   q =  0.038097
Atom  5 N   q =  0.495755
Atom  0 N   q =  0.495797


In [36]:
# Cell 7: Save the optimized coordinates
output_xyz = 'dft_optimized_final.xyz'
mol_eq.tofile(output_xyz)
print(f"Saved to /kaggle/working/{output_xyz}")
print("Download it from the Output tab on the right panel.")

Saved to /kaggle/working/dft_optimized_final.xyz
Download it from the Output tab on the right panel.
